# Multi-Agent Workflow for Geophysical Data Processing

**Automatic Cross-Modal Geophysics Agent for Subsurface Hydrology**

This notebook demonstrates how to use the cross-modal geophysics agent system for subsurface hydrology to automate complete workflows, processing geophysical data (ERT, seismic, etc.) into hydrologic information with climate data integration.

The system supports multiple LLM APIs (OpenAI GPT, Google Gemini, Anthropic Claude) and can handle various geophysical data types.

**Workflow:** "load geophysical data → fetch climate data → process → invert → convert to hydrologic parameters → report with cross-modal climate reasoning"

## Agent Overview

Each agent is specialized for a specific task:

1. **ClimateDataAgent**: Fetches climate data (precipitation, temperature, PET) for cross-modal reasoning
2. **ERTLoaderAgent**: Loads and quality-checks ERT data
3. **SeismicAgent** (optional): Processes seismic data for structural constraints
4. **ERTInversionAgent**: Performs ERT inversion
5. **WaterContentAgent**: Converts resistivity to water content with uncertainty
6. **ReportAgent**: Generates comprehensive reports with climate-based resistivity interpretation

The **AgentCoordinator** manages the workflow and ensures proper data flow between agents, enabling cross-modal reasoning where climate features explain resistivity changes (e.g., post-rainfall decreases, drying during high PET).

In [ ]:
import os
import sys

# Setup package path
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from PyHydroGeophysX.agents import (
    AgentCoordinator,
    ERTLoaderAgent,
    ERTInversionAgent,
    WaterContentAgent,
    ReportAgent,
    SeismicAgent,
    ClimateDataAgent
)

API path =  c:\Users\hchen117\.conda\envs\pg\Lib\site-packages\resipy
ResIPy version =  3.6.3
cR2.exe found and up to date.
R3t.exe found and up to date.
cR3t.exe found and up to date.
API path =  c:\Users\hchen117\.conda\envs\pg\Lib\site-packages\resipy
ResIPy version =  3.6.3
cR2.exe found and up to date.
R3t.exe found and up to date.
cR3t.exe found and up to date.


## Setup: Configure LLM API Key

The system supports multiple LLM providers. Set the appropriate API key as an environment variable:

- **OpenAI**: `export OPENAI_API_KEY='your-key'`
- **Google Gemini**: `export GEMINI_API_KEY='your-key'`
- **Anthropic Claude**: `export ANTHROPIC_API_KEY='your-key'`

Or set it directly in the code below (not recommended for shared notebooks).

In [2]:
# Select LLM provider: 'openai', 'gemini', or 'claude'
llm_provider = 'openai'  # Change this to use different LLM provider

# Get API key based on provider
provider_env_map = {
    'openai': 'OPENAI_API_KEY',
    'gemini': 'GEMINI_API_KEY',
    'claude': 'ANTHROPIC_API_KEY'
}
api_key = os.getenv(provider_env_map.get(llm_provider, 'OPENAI_API_KEY'))

# Alternatively, set it directly (not recommended for shared notebooks):
# api_key = 'your-api-key-here'

if not api_key:
    print(f"⚠️  Warning: API key for {llm_provider} not found in environment variables.")
    print("   The system will work but LLM-enhanced features will be disabled.")
    print(f"   Set it with: os.environ['{provider_env_map[llm_provider]}'] = 'your-key'")
else:
    print(f"✓ API key for {llm_provider} found!")

⚠️  Warning: API key for openai not found in environment variables.
   The system will work but LLM-enhanced features will be disabled.
   Set it with: os.environ['OPENAI_API_KEY'] = 'your-key'


## Example 1: Standard ERT Workflow

This example demonstrates the complete workflow:
1. Initialize agent coordinator
2. Register specialized agents
3. Configure workflow parameters
4. Execute workflow
5. Access and display results

In [ ]:
print("=" * 80)
print("Multi-Agent Geophysical Workflow Example")
print("Automatic Cross-Modal Geophysics Agent for Subsurface Hydrology")
print("=" * 80)

# Step 1: Initialize the coordinator
print("\n[1/5] Initializing agent coordinator...")
coordinator = AgentCoordinator(
    api_key=api_key,
    output_dir="results/agents_workflow",
    llm_provider=llm_provider  # Supports 'openai', 'gemini', 'claude'
)
print("   ✓ Coordinator initialized")

In [ ]:
# Step 2: Register specialized agents
print("[2/5] Registering specialized agents...")

# Register climate data agent for cross-modal analysis
climate_agent = ClimateDataAgent(api_key=api_key, llm_provider=llm_provider)
coordinator.register_agent('climate_data', climate_agent)

# Register ERT loader agent
ert_loader = ERTLoaderAgent(api_key=api_key, llm_provider=llm_provider)
coordinator.register_agent('ert_loader', ert_loader)

# Register ERT inversion agent
ert_inversion = ERTInversionAgent(api_key=api_key, llm_provider=llm_provider)
coordinator.register_agent('ert_inversion', ert_inversion)

# Register water content conversion agent
water_content = WaterContentAgent(api_key=api_key, llm_provider=llm_provider)
coordinator.register_agent('water_content', water_content)

# Register report generation agent
report_gen = ReportAgent(api_key=api_key, llm_provider=llm_provider)
coordinator.register_agent('report', report_gen)

# Register seismic agent (optional)
seismic = SeismicAgent(api_key=api_key, llm_provider=llm_provider)
coordinator.register_agent('seismic_processor', seismic)

print("   ✓ All agents registered")

In [ ]:
# Step 3: Configure workflow
print("[3/5] Configuring workflow...")

workflow_config = {
    # Data source configuration
    'data_file': 'data/ERT/E4D/2021-10-08_1400.ohm',
    'project_dir': 'data/ERT/E4D',
    'instrument': 'E4D',
    'crs': 'local',
    
    # Climate data integration for cross-modal reasoning
    'use_climate': True,
    'climate_config': {
        # Site coordinates (longitude, latitude) for climate data retrieval
        # TODO: Update these coordinates to match your actual site location!
        # Example: Adjust to your site location
        'coords': (-105.3, 40.0),  # Example: Colorado location (MUST BE UPDATED)
        'dates': ('2021-09-01', '2021-11-30'),  # Date range covering ERT campaign
        'crs': 4326,  # WGS84 for coordinates
        'variables': ['prcp', 'tmin', 'tmax', 'srad', 'vp', 'dayl'],
        'pet_method': 'penman_monteith',  # or 'priestley_taylor', 'hargreaves_samani'
        'pet_params': {
            'arid_correction': False,  # Set True for arid regions
        },
        'time_scale': 'daily',
        'region': 'na',  # North America
        'antecedent_days': [1, 3, 7, 14],  # Antecedent precipitation windows
    },
    # ERT acquisition timestamps for climate alignment
    # TODO: Update these timestamps to match your actual ERT acquisition times!
    'ert_timestamps': [
        '2021-10-08',  # Example timestamp (MUST BE UPDATED)
        # Add more timestamps as needed for your campaign
    ],
    
    # Inversion parameters
    'inversion_params': {
        'lambda': 20.0,          # Regularization parameter
        'max_iterations': 10,     # Maximum inversion iterations
        'method': 'cgls',         # Solver method
        'use_gpu': False          # GPU acceleration (requires CuPy)
    },
    
    # Petrophysical parameters for water content conversion
    'petrophysical_params': {
        # Parameters will be auto-suggested by LLM if not provided
        # or can be manually specified per layer
        # Example:
        # 0: {  # Layer marker 0 (top layer)
        #     'rhos': {'mean': 100.0, 'std': 20.0},
        #     'n': {'mean': 2.2, 'std': 0.2},
        #     'sigma_sur': {'mean': 0.002, 'std': 0.0005},
        #     'porosity': {'mean': 0.40, 'std': 0.05}
        # }
    },
    
    # Uncertainty quantification
    'run_uncertainty': True,      # Run Monte Carlo analysis
    'n_realizations': 100,        # Number of MC realizations
    
    # Optional: Seismic integration
    'use_seismic': False,         # Set to True to use seismic constraints
    # 'seismic_data': seismic_travel_time_data,  # Provide seismic data if available
    # 'velocity_threshold': 1200,  # m/s threshold for interface detection
}

print("   ✓ Workflow configured")

In [ ]:
# Step 4: Execute workflow
print("[4/5] Executing workflow...")
print("-" * 80)

try:
    results = coordinator.execute_workflow(workflow_config)
    
    print("-" * 80)
    print("[5/5] Workflow completed!")
    print(f"\n✓ Status: {results['status']}")
    
    # Display summary
    if results['status'] == 'success':
        print("\n📊 Workflow Summary:")
        summary = coordinator.get_workflow_summary()
        print(f"   - Completed steps: {', '.join(summary['completed_steps'])}")
        print(f"   - Available results: {', '.join(summary['available_results'])}")
        
        # Access specific results
        if 'water_content' in results['results']:
            wc_results = results['results']['water_content']
            print(f"\n💧 Water Content Results:")
            print(f"   - Output directory: {wc_results['output_dir']}")
            if wc_results.get('interpretation'):
                print(f"   - Interpretation: {wc_results['interpretation']}")
        
        if 'report' in results['results']:
            report_results = results['results']['report']
            print(f"\n📄 Report Generated:")
            print(f"   - Report file: {report_results['report_file']}")
            if report_results.get('html_file'):
                print(f"   - HTML report: {report_results['html_file']}")
    
    else:
        print(f"\n❌ Workflow failed: {results.get('error', 'Unknown error')}")
        if 'partial_results' in results:
            print(f"   Partial results available for: {list(results['partial_results'].keys())}")
    
except Exception as e:
    print(f"\n❌ Error executing workflow: {str(e)}")
    import traceback
    traceback.print_exc()
    results = None

## Example 2: Cross-Modal Geophysical Workflow with Seismic Integration

This example shows how to include seismic refraction data for structure-constrained ERT inversion using the cross-modal geophysics agent system.

**Note:** Uncomment and run the cell below to execute the seismic-integrated workflow.

In [ ]:
# Uncomment to run seismic-integrated workflow

print("=" * 80)
print("Multi-Agent Cross-Modal Geophysical Workflow with Seismic Integration")
print("=" * 80)

# Initialize coordinator
coordinator_seismic = AgentCoordinator(
    api_key=api_key, 
    output_dir="results/agents_seismic",
    llm_provider=llm_provider
)

# Register all agents (including climate and seismic) - consistent with main example
coordinator_seismic.register_agent('climate_data', ClimateDataAgent(api_key=api_key, llm_provider=llm_provider))
coordinator_seismic.register_agent('ert_loader', ERTLoaderAgent(api_key=api_key, llm_provider=llm_provider))
coordinator_seismic.register_agent('seismic_processor', SeismicAgent(api_key=api_key, llm_provider=llm_provider))
coordinator_seismic.register_agent('ert_inversion', ERTInversionAgent(api_key=api_key, llm_provider=llm_provider))
coordinator_seismic.register_agent('water_content', WaterContentAgent(api_key=api_key, llm_provider=llm_provider))
coordinator_seismic.register_agent('report', ReportAgent(api_key=api_key, llm_provider=llm_provider))

# Configure workflow with seismic and climate integration
workflow_config_seismic = {
    'data_file': 'data/ERT/E4D/2021-10-08_1400.ohm',
    'project_dir': 'data/ERT/E4D',
    'instrument': 'E4D',
    'crs': 'local',
    
    # Enable climate integration for cross-modal reasoning
    'use_climate': True,
    'climate_config': {
        'coords': (-105.3, 40.0),  # Adjust to your site location
        'dates': ('2021-09-01', '2021-11-30'),
        'crs': 4326,
        'variables': ['prcp', 'tmin', 'tmax', 'srad', 'vp', 'dayl'],
        'pet_method': 'penman_monteith',
        'pet_params': {'arid_correction': False},
        'time_scale': 'daily',
        'region': 'na',
        'antecedent_days': [1, 3, 7, 14],
    },
    'ert_timestamps': ['2021-10-08'],
    
    # Enable seismic integration
    'use_seismic': True,
    'seismic_data': None,  # Provide seismic travel time data object
    'velocity_threshold': 1200,  # m/s
    
    'inversion_params': {
        'lambda': 20.0,
        'max_iterations': 10,
    },
    
    'run_uncertainty': True,
    'n_realizations': 50,
}

print("\n🔬 Running workflow with seismic constraints...")

try:
    results_seismic = coordinator_seismic.execute_workflow(workflow_config_seismic)
    
    if results_seismic['status'] == 'success':
        print("\n✓ Workflow with seismic integration completed successfully!")
        
        # Check if seismic results are available
        if 'seismic_structure' in results_seismic['results']:
            seis = results_seismic['results']['seismic_structure']
            print(f"\n🌊 Seismic Results:")
            print(f"   - Interface extracted: Yes")
            print(f"   - Velocity threshold: {seis['velocity_threshold']} m/s")
            if seis.get('interpretation'):
                print(f"   - Interpretation: {seis['interpretation']}")

except Exception as e:
    print(f"\n❌ Error: {str(e)}")

## Summary

The **automatic cross-modal geophysics agent system for subsurface hydrology** provides a flexible and automated way to process geophysical data into hydrologic information.

### Key Benefits:

- **Cross-Modal Integration**: Seamlessly combines different geophysical methods (ERT, seismic)
- **Multiple LLM Support**: Works with OpenAI GPT, Google Gemini, or Anthropic Claude
- **Modular Design**: Each agent handles a specific task
- **LLM-Enhanced Intelligence**: Automatic parameter suggestion and interpretation
- **Flexible Configuration**: Easy to customize workflows
- **Robust Error Handling**: Manages errors gracefully with partial results
- **Extensible**: Easy to add new agents for custom workflows

### Supported LLM Providers:

- **OpenAI GPT**: gpt-4, gpt-3.5-turbo, gpt-4-turbo
- **Google Gemini**: gemini-pro, gemini-1.5-pro
- **Anthropic Claude**: claude-3-opus, claude-3-sonnet, claude-3-haiku

For more examples and documentation, visit: https://geohang.github.io/PyHydroGeophysX/